In [ ]:
%load_ext rich

In [ ]:
import json
import os
from pathlib import Path

import requests
from tqdm import tqdm

from aymurai.experiments.entity_disambiguation.runner import (
    call_extraction_api as extract_document,
)

API_URL = "http://localhost:8999"  # Url for debugger. change it to your own

## Sample document


In [ ]:
doc_path = Path(
    "/resources/data/restricted/disambiguation-eval/documents/aymurai - ejemplo 02.docx"
)

## /document-extract endpoint output


In [ ]:
# /document-extract endpoint output
session = requests.Session()
document = extract_document(
    session,
    endpoint=f"{API_URL}/misc/document-extract",
    file_path=doc_path,
    timeout_s=300,
)
paragraphs = document["detail"]["document"]
document

In [ ]:
len(paragraphs)

## Inference


In [ ]:
# Function to make inference using the API
def get_predictions(sample: str) -> dict:
    response = requests.post(url=f"{API_URL}/anonymizer/predict", json={"text": sample})
    response.raise_for_status()
    return response.json()

In [ ]:
predictions = [get_predictions(paragraph) for paragraph in tqdm(paragraphs)]
predictions

## Export variants


In [ ]:
def disambiguate_and_export(
    variant_name: str, label_policies: dict, render_policy: dict
):
    response = requests.post(
        url=f"{API_URL}/anonymizer/disambiguate",
        json={
            "paragraphs": predictions,
            # "custom_prompts": {"root": []},
            "label_policies": label_policies,
        },
    )
    response.raise_for_status()
    disambiguated = response.json()

    json_prediction = json.dumps(
        {
            "data": disambiguated["data"],
            "label_policies": label_policies,
            "render_policy": render_policy,
        }
    )

    with open(doc_path, "rb") as file:
        files = {"file": file}

        response = requests.post(
            url=f"{API_URL}/anonymizer/anonymize-document",
            data={"annotations": json_prediction},
            files=files,
        )
        response.raise_for_status()

    output_dir = "output"
    os.makedirs(output_dir, exist_ok=True)

    filename = os.path.basename(doc_path)
    filename, ext = os.path.splitext(filename)
    out_path = f"{output_dir}/{filename}-{variant_name}.odt"

    with open(out_path, "wb") as file:
        file.write(response.content)

    return out_path

In [ ]:
render_policy = {
    "use_subclass_when_available": True,
    "fallback_to_label": True,
    "suffix_mode": "auto",
    "suffix_threshold": 1,
}

# 1) everything fuzzy
label_policies_all_fuzzy = {
    "PER": {"disambiguation": "fuzzy", "anonymize": True},
    "DNI": {"disambiguation": "fuzzy", "anonymize": True},
    "LOC": {"disambiguation": "fuzzy", "anonymize": True},
    "DIRECCION": {"disambiguation": "fuzzy", "anonymize": True},
    "FECHA": {"disambiguation": "fuzzy", "anonymize": True},
}
out_all_fuzzy = disambiguate_and_export(
    "all-fuzzy", label_policies_all_fuzzy, render_policy
)

# 2) FECHAs excluded
label_policies_no_fecha = {
    "PER": {"disambiguation": "fuzzy", "anonymize": True},
    "DNI": {"disambiguation": "fuzzy", "anonymize": True},
    "LOC": {"disambiguation": "fuzzy", "anonymize": True},
    "DIRECCION": {"disambiguation": "fuzzy", "anonymize": True},
    "FECHA": {"disambiguation": "none", "anonymize": False},
}
out_no_fecha = disambiguate_and_export(
    "no-fecha", label_policies_no_fecha, render_policy
)

# 3) PER llm-disambiguated
label_policies_per_llm = {
    "PER": {"disambiguation": "llm", "anonymize": True},
    "DNI": {"disambiguation": "fuzzy", "anonymize": True},
    "LOC": {"disambiguation": "fuzzy", "anonymize": True},
    "DIRECCION": {"disambiguation": "fuzzy", "anonymize": True},
    "FECHA": {"disambiguation": "none", "anonymize": False},
}
out_per_llm = disambiguate_and_export("per-llm", label_policies_per_llm, render_policy)

out_all_fuzzy, out_no_fecha, out_per_llm